In [ ]:
import numpy as np
import torch

from rlaopt.atoms import L1Norm, SumSquares
from rlaopt.atoms.polyhedra import Polyhedra
from rlaopt.expression.bilevel_expression import BilevelExpression
from rlaopt.expression.expression import Variable
from rlaopt.operator_split import OperatorSplit
from rlaopt.solvers.prox_grad import ProxGrad, ProxGradConfig

In [ ]:
A = torch.randn(100, 10)
C = torch.randn(20, 10)
x_fes = torch.randn(
    10,
)
b = A @ x_fes
l = torch.min(C @ x_fes) * torch.ones(20)
u = torch.max(C @ x_fes) * torch.ones(20)

In [ ]:
x = Variable(torch.ones(10))

In [ ]:
f = 0.5 * ((A @ x - b) ** 2).sum()

In [ ]:
f

In [ ]:
P = Polyhedra(x, A=A, b=b, C=C, l=l, u=u)

In [ ]:
(A @ x_fes == b).all()

In [ ]:
P.evaluate_at(x=x_fes)

In [ ]:
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

In [ ]:
# Low-rank matrix completion data generation
Xstar = torch.randn(1000, 10) / (1000**0.5)
Ystar = torch.randn(500, 10) / (500**0.5)
A = Xstar @ Ystar.T

In [ ]:
# Initialize variables
X, Y = (
    Variable(torch.randn(1000, 10) / (1000**0.5)),
    Variable(torch.randn(500, 10) / (500**0.5)),
)

In [ ]:
# Setup the objective function
obj = SumSquares(X @ Y.T - A)

In [ ]:
# Test objective function evaluation
obj_val = obj.forward()
obj_true = torch.linalg.norm(X.value @ Y.value.T - A, ord="fro") ** 2
print("Error in computed Objective value:", abs(obj_val - obj_true))

In [ ]:
# Setup APG with linesearch
config = ProxGradConfig(eta=1.0, tol=1e-6, use_acceleration=True, use_linesearch=True)
opt = ProxGrad(config, obj)

In [ ]:
# Solve with APG step method
params = obj.params
state = opt.init_state(params)
for i in range(200):
    params, state = opt.step(params, state)
    if i % 10 == 0:
        print(f"Iteration {i}, Objective Value: {obj.evaluate(params)}")
print(state.err)

In [ ]:
names = list(params.keys())

In [ ]:
torch.linalg.norm(params[names[0]] @ params[names[1]].T - A, ord="fro") ** 2

In [2]:
# Lasso data generation
n, p = 1024, 128
ntst = 256
s = 32
J = np.random.choice(p, s)

xStar = torch.zeros(p)
xStar[J] = torch.randn(s) / (s**0.5)
A = torch.randn(n, p) / (n**0.5)
Atst = torch.randn(ntst, p) / (ntst**0.5)
b = A @ xStar + 0.001 * torch.randn(n)
btst = Atst @ xStar + 0.001 * torch.randn(ntst)

In [3]:
# Init params + reg
x = Variable(torch.zeros(p))
mu = 0.1 * torch.linalg.norm(A.T @ b, ord=torch.inf)

In [4]:
# Lasso problem
F = SumSquares(A @ x - b) + L1Norm(x, scaling=mu)

In [6]:
f, r = F.operator_split()

In [ ]:
F.forward()

In [ ]:
torch.linalg.norm(A @ x.value - b, 2) ** 2 + mu * torch.linalg.norm(x.value, ord=1)

In [ ]:
# Step size is reciprocal of Lipschitz constant
eta = 1 / (2 * torch.linalg.norm(A, ord=2) ** 2)

In [ ]:
config = ProxGradConfig(eta=eta, tol=1e-6, use_acceleration=True, use_linesearch=False)
opt = ProxGrad(config, F)

In [ ]:
# Solve the problem using step method
params = F.params
state = opt.init_state(params)
for i in range(100):
    print(state.err)
    params, state = opt.step(params, state)
print(state.err)

In [ ]:
# Solve the problem using solve method
params, err = opt.solve(F)
# Print norm of gradient mapping
print(err)

In [ ]:
params

In [ ]:
# Expected to be the same as at initialization, since the optimizer does not update in-place
x.value

In [ ]:
# Test OperatorSplit interface with the same problem
obj = OperatorSplit(SumSquares(A @ x - b), L1Norm(x, scaling=mu))

In [ ]:
config = ProxGradConfig(eta=eta, tol=1e-6, use_acceleration=True, use_linesearch=False)
opt = ProxGrad(config, obj)

In [ ]:
# Solve the problem using step method
params = obj.f.params
state = opt.init_state(params)
for i in range(100):
    print(state.err)
    params, state = opt.step(params, state)
print(state.err)

In [ ]:
# Solve the problem using solve method
params, err = opt.solve(obj)
# Print norm of gradient mapping
print(err)

In [ ]:
# Test HVP computation
v = torch.randn(p)
Hv = obj.hvp_f(params, v)
print("HVP error:", torch.linalg.norm(Hv - 2 * (A.T @ (A @ v))))

In [ ]:
# Bilevel Expression for optimizing lasso regularization parameter

# Inner function: Lasso on training data
ftr = lambda mu: SumSquares(A @ x - b) + L1Norm(x, mu)

# Outer function: Least squares on test data
ftst = SumSquares(Atst @ x - btst)

# Setup bilevel problem
obj = BilevelExpression(mu, ftr, ftst, config, ProxGrad)

In [ ]:
# Setup solver for the bilevel problem
config_blvl = ProxGradConfig(
    eta=torch.tensor(0.001),
    max_iters=500,
    tol=1e-3,
    use_acceleration=False,
    use_linesearch=False,
)
opt_blvl = ProxGrad(config_blvl, obj)

In [ ]:
# Initial objective value
obj.forward()

In [ ]:
# Solve the bilevel problem using solve method
mu_star, err = opt_blvl.solve(obj)

In [ ]:
# Final objective value
obj.evaluate(mu_star)

In [ ]:
# Print the optimal regularization parameter
mu_star

In [ ]:
# Print gradient norm of the objective at mu_star
torch.func.grad(obj.evaluate)(mu_star)["w"].norm().item()